In [1]:
import pandas as pd
import japanize_matplotlib
from matplotlib import pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import to_rgba
import numpy as np
import glob
import os
import re
from enum import Enum

class Area(Enum):
    YOKOSUKA = "Yokosuka"

class ModelType(Enum):
    AGENTS_MODEL = "AgentsModel"
    EVACUEE_ONLY = "EvacueesOnly"
    
class Mode(Enum):
    TRAIN = "Train"
    INFERENCE = "Inference"

In [28]:
# 各種CSVデータの読み込み部
# 条件指定 TODO : enumで簡単に指定できるようにする

DATAFOLDER = "../../Master-Simulator/Assets/Data_2025-01-10_01-09-42-Case3"
SIMULATE_ID = "474dc0ca-77d1-458d-bdc1-2378a37116d2"
AREA = Area.YOKOSUKA
MODELTYPE = ModelType.EVACUEE_ONLY
MODE = Mode.INFERENCE


# データ読み込み
AGENT_LOG_FOLDER = "AgentActionLogs"
ENV_EVACUATERATE_FOLDER = "EnvEvacuationRate"
SHELTER_COUNTLOG_FOLDER = "TowerEvacueeCount"
SUMMURY_FILENAME = "EpisodeSummary"

agent_action_Log_df: pd.DataFrame
env_evacuaterate_df: pd.DataFrame
shelter_countlog_df: pd.DataFrame

# エージェントの行動ログの読み込み
if(MODELTYPE == ModelType.AGENTS_MODEL):
    folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{AGENT_LOG_FOLDER}"
    csv_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
    dataframes = []
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        parent_folder = os.path.basename(os.path.dirname(file_path))
        match = re.search(r'Ep-(\d+)', file_name)
        episode_num = int(match.group(1)) if match else None
        df = pd.read_csv(file_path)
        df['AgentID'] = parent_folder
        df["Episode"] = episode_num
        dataframes.append(df)

    agent_action_Log_df = pd.concat(dataframes, ignore_index=True)

# 環境の避難率の読み込み
folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{ENV_EVACUATERATE_FOLDER}"
csv_files = glob.glob(os.path.join(folder_path, '*.csv'), recursive=True)
dataframes = []
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    # Episode番号を取得
    match = re.search(r'Ep-(\d+)', file_name)
    episode_num = int(match.group(1)) if match else None
    df["Episode"] = episode_num
    dataframes.append(df)

env_evacuaterate_df = pd.concat(dataframes, ignore_index=True)

# 避難所毎の避難者数推移の読み込み
folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{SHELTER_COUNTLOG_FOLDER}"
csv_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
dataframes = []
for file_path in csv_files:
    parent_folder = os.path.basename(os.path.dirname(file_path))
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    df['ShelterID'] = parent_folder
    match = re.search(r'Ep-(\d+)', file_name)
    episode_num = int(match.group(1)) if match else None
    dataframes.append(df)
    df["Episode"] = episode_num
    dataframes.append(df)

shelter_countlog_df = pd.concat(dataframes, ignore_index=True)

summary_df = pd.read_csv(f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{AREA.value}_{MODELTYPE.value}_{SUMMURY_FILENAME}.csv")


### エピソード毎の概要データ
| 列名                   | 説明                                                                 |
|------------------------|----------------------------------------------------------------------|
| `index`                | エピソード番号。各エピソードの識別子として使用されます。             |
| `Limit Time`           | 制限時間（秒）。各エピソードの最大許容時間を示します。               |
| `End Time Sec`         | エピソードが終了した経過時間（秒）。エピソードの実際の終了時間です。 |
| `Total Evacuee Count`  | スポーンした避難者の総数。シミュレーション中に生成された避難者の数。 |
| `Drone Count`          | ドローンの総数。シミュレーションに参加したドローンの数。             |
| `Final Evacuate Rate`  | 最終的な避難完了率。避難者のうち、最終的に避難を完了した割合。       |


In [ ]:
summary_df


In [ ]:
# 制限時間のヒストグラム
plt.hist(summary_df["Limit Time"])
plt.xlabel("Limit Time")
plt.ylabel("Frequency")
plt.title("Limit Time Hist")

In [ ]:
# エピソード毎の最終避難率のプロット
plt.figure(figsize=(10, 6))
plt.plot(summary_df.index, summary_df['Final Evacuate Rate'], marker='o')
plt.xlabel('Episode Number')
plt.ylabel('Final Evacuate Rate')
plt.title('Final Evacuate Rate per Episode')
plt.grid(True)
plt.show()

In [ ]:
# 避難率のヒストグラム
plt.hist(summary_df["Final Evacuate Rate"])
plt.xlabel("Final Evacuate Rate")
plt.ylabel("Frequency")
plt.title("Final Evacuate Rate Hist")

### エージェントの行動ログデータ
| 列名               | 説明                                                                 |
|--------------------|----------------------------------------------------------------------|
| `Elapsed Sec`      | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Destination`      | エージェントが選択した避難所。エージェントが向かっている避難所の名称。|
| `Speed`            | エージェントが選択した移動速度。エージェントの現在の移動速度を示します。|
| `Guided Evacuees`  | エージェントが誘導中の避難者数。現在エージェントが誘導している避難者の数。|
| `AgentID`          | エージェントID。各エージェントの一意の識別子。                       |
| `Episode`          | エピソード番号。シミュレーションのエピソードを識別する番号。         |


In [ ]:
agent_action_Log_df = agent_action_Log_df.sort_values(by=['AgentID', 'Episode', 'Elapsed Sec'])
agent_action_Log_df

In [ ]:
agents = agent_action_Log_df['AgentID'].unique()

for agent in agents:
    plt.figure(figsize=(12, 8))
    agent_data = agent_action_Log_df[agent_action_Log_df['AgentID'] == agent]
    agent_data['Destination'].value_counts().plot(kind='bar')
    
    plt.xlabel('Destination')
    plt.ylabel('Frequency')
    plt.title(f'Destination Distribution for Agent {agent}')
    plt.grid(True)
    plt.show()

In [ ]:
# エージェントごとの誘導中人数の推移をプロット
agents = agent_action_Log_df['AgentID'].unique()

for agent in agents:
    plt.figure(figsize=(12, 8))
    agent_data = agent_action_Log_df[agent_action_Log_df['AgentID'] == agent]
    
    # Get unique episodes for the agent
    episodes = agent_data['Episode'].unique()
    
    # Define a colormap
    colors = cm.get_cmap('tab20', len(episodes))
    
    # Plot each episode for the agent
    for i, episode in enumerate(episodes):
        episode_data = agent_data[agent_data['Episode'] == episode]
        plt.plot(episode_data['Elapsed Sec'], episode_data['Guided Evacuees'], label=f'Episode {episode}', color=colors(i))
    
    plt.xlabel('Elapsed Sec')
    plt.ylabel('Guided Evacuees')
    plt.title(f'Guided Evacuees per Episode for Agent {agent}')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.grid(True)
    plt.show()

### 環境全体の避難率推移データ
| 列名            | 説明                                                                 |
|-----------------|----------------------------------------------------------------------|
| `Elapsed Sec`   | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Evacuate Rate` | 避難率。避難者のうち、避難を完了した割合を示します。                 |
| `Episode`       | エピソード番号。各シミュレーションの識別子として使用されます。       |


In [ ]:
# エピソード番号と経過時間でソート
env_evacuaterate_df = env_evacuaterate_df.sort_values(by=['Episode', 'Elapsed Sec'])
env_evacuaterate_df

In [ ]:
# エピソード毎の避難率推移のプロット
plt.figure(figsize=(12, 8))

# Get unique episodes
episodes = env_evacuaterate_df['Episode'].unique()

# Define a colormap
colors = cm.get_cmap('tab20', len(episodes))

# Plot each episode
for i, episode in enumerate(episodes):
    episode_data = env_evacuaterate_df[env_evacuaterate_df['Episode'] == episode]
    plt.plot(episode_data['Elapsed Sec'], episode_data['Evacuation Rate'], label=f'Episode {episode}', color=colors(i))

plt.xlabel('Elapsed Sec')
plt.ylabel('Evacuation Rate')
plt.title('Evacuation Rate per Episode')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True)
plt.show()

### 避難所毎の避難者数推移データ
| 列名            | 説明                                                                 |
|-----------------|----------------------------------------------------------------------|
| `Elapsed Sec`   | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Evacuate Rate` | 避難率。避難者のうち、避難を完了した割合を示します。                 |
| `Episode`       | エピソード番号。各シミュレーションの識別子として使用されます。       |


In [ ]:
shelter_countlog_df = shelter_countlog_df.sort_values(by=['ShelterID', 'Episode', 'Elapsed Sec'])
shelter_countlog_df

In [ ]:
# 避難所ごとの累積避難者数の推移をプロット
shelters = shelter_countlog_df['ShelterID'].unique()

for shelter in shelters:
    plt.figure(figsize=(12, 8))
    shelter_data = shelter_countlog_df[shelter_countlog_df['ShelterID'] == shelter]
    
    # Get unique episodes for the shelter
    episodes = shelter_data['Episode'].unique()
    
    # Define a colormap
    colors = cm.get_cmap('tab20', len(episodes))
    
    # Plot each episode for the shelter
    for i, episode in enumerate(episodes):
        episode_data = shelter_data[shelter_data['Episode'] == episode]
        plt.plot(episode_data['Elapsed Sec'], episode_data['Evacuee Count'], label=f'Episode {episode}', color=colors(i))
    
    plt.xlabel('Elapsed Sec')
    plt.ylabel('Evacuee Count')
    plt.title(f'Evacuee Count per Episode for Shelter {shelter}')
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.grid(True)
    plt.show()